# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

In [1]:
import os
import re
from nltk.stem import PorterStemmer
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [2]:
path = "/kaggle/input/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects"
archivo = os.path.join(path, "wikipedia_text_corpus.csv")

df = pd.read_csv(archivo)

print(df.head())
print(df.columns)

   Unnamed: 0                                               text
0           1  Anovo\n\nAnovo (formerly A Novo) is a computer...
1           2  Battery indicator\n\nA battery indicator (also...
2           3  Bob Pease\n\nRobert Allen Pease (August 22, 19...
3           4  CAVNET\n\nCAVNET was a secure military forum w...
4           5  CLidar\n\nThe CLidar is a scientific instrumen...
Index(['Unnamed: 0', 'text'], dtype='object')


In [3]:
corpus = df["text"].dropna().tolist()
nombres = [f"doc_{i}" for i in range(len(corpus))]

print("Documentos cargados:", len(corpus))

Documentos cargados: 10859


In [4]:
def procesar(texto):
    texto = texto.lower()
    doc = re.sub(r'[^a-z\s]', ' ', texto)
    doc = re.sub(r'\s+', ' ', doc).strip()
    return doc

In [5]:
corpus_limpio = [procesar(doc) for doc in corpus]

In [6]:
def tokenizar(texto):
    return texto.split()

In [7]:
corpus_tokenizado = [tokenizar(doc) for doc in corpus_limpio]

In [8]:
stemmer = PorterStemmer()

def aplicar_stemming(tokens):
    return [stemmer.stem(p) for p in tokens]

In [9]:
corpus_stemming = [aplicar_stemming(doc) for doc in corpus_tokenizado]

In [10]:
corpus_final = [' '.join(doc) for doc in corpus_stemming]

## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [11]:
df_corpus = pd.DataFrame({
    'raw': corpus,
    'processed': corpus_final
})

In [12]:
vectorizador = TfidfVectorizer(stop_words='english')

In [13]:
tfidf_matrix = vectorizador.fit_transform(df_corpus['processed'])

In [14]:
def procesar_query(query):
    query = procesar(query)
    query_tokens = tokenizar(query)
    query_stem = aplicar_stemming(query_tokens)
    return ' '.join(query_stem)

In [15]:
def buscar_tfidf(query, top_n=5):
    query_procesada = procesar_query(query)
    query_vector = vectorizador.transform([query_procesada])
    similitudes = cosine_similarity(query_vector, tfidf_matrix).flatten()
    indices = similitudes.argsort()[-top_n:][::-1]
    return indices, similitudes

In [16]:
query = "science and technology"

indices, similitudes = buscar_tfidf(query)

for i in indices:
    print("Documento:", nombres[i])
    print("Score:", similitudes[i])
    print(corpus[i][:300])
    print("-" * 80)

Documento: doc_3842
Score: 0.6575076298916748
Ministry of Science and Technology (China)

The Ministry of Science and Technology (MOST) of the People's Republic of China, formerly the State Science and Technology Commission, is the central government ministry which coordinates science and technology activities in the country.

It succeeded the 
--------------------------------------------------------------------------------
Documento: doc_1317
Score: 0.5823675489622095
Ministry of Science and Technology (Taiwan)

The Ministry of Science and Technology (MOST; ) is the government ministry of the Republic of China (Taiwan) for the promotion and funding of academic research, development of science and technology and science parks. MOST is a member of Belmont Forum.

T
--------------------------------------------------------------------------------
Documento: doc_6377
Score: 0.5109235001938691
Department of Science and Technology (Philippines)

The Philippines' Department of Science and Tec

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [17]:
corpus_bm25 = [doc.split() for doc in corpus_final]

In [18]:
doc_lengths = [len(doc) for doc in corpus_bm25]

avg_doc_length = sum(doc_lengths) / len(doc_lengths)

print("Longitud promedio:", avg_doc_length)

Longitud promedio: 761.9451146514411


In [19]:
from collections import Counter

tf_docs = []

for doc in corpus_bm25:
    tf_docs.append(Counter(doc))

In [20]:
df = {}

for doc in corpus_bm25:

    unique_terms = set(doc)

    for term in unique_terms:
        df[term] = df.get(term, 0) + 1

In [21]:
import math

N = len(corpus_bm25)

idf = {}

for term, freq in df.items():
    idf[term] = math.log((N - freq + 0.5) / (freq + 0.5) + 1)

In [22]:
k1 = 1.5
b = 0.75

In [23]:
def bm25_score(query_tokens, doc_index):

    score = 0

    doc = corpus_bm25[doc_index]

    doc_tf = tf_docs[doc_index]

    doc_length = doc_lengths[doc_index]

    for term in query_tokens:

        if term not in doc_tf:
            continue

        term_freq = doc_tf[term]

        term_idf = idf.get(term, 0)

        numerator = term_freq * (k1 + 1)

        denominator = term_freq + k1 * (
            1 - b + b * (doc_length / avg_doc_length)
        )

        score += term_idf * (numerator / denominator)

    return score

In [24]:
def buscar_bm25_manual(query, top_n=5):

    query_procesada = procesar_query(query)

    query_tokens = query_procesada.split()

    scores = []

    for i in range(len(corpus_bm25)):

        score = bm25_score(query_tokens, i)

        scores.append(score)

    ranked = sorted(
        enumerate(scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:top_n]

In [25]:
query = "science and technology"

resultados = buscar_bm25_manual(query)

for idx, score in resultados:

    print("Documento:", nombres[idx])

    print("Score:", score)

    print(corpus[idx][:300])

    print("-" * 80)

Documento: doc_5501
Score: 7.42082551374947
Ministry of Science and Technology (Pakistan)

The Ministry of Science and Technology (, abbreviated as MoST) is a Cabinet-level ministry of the Government of Pakistan concerned with science and technology in Pakistan and in general, Pakistan's science policy, planning, co-ordination and directing o
--------------------------------------------------------------------------------
Documento: doc_6377
Score: 7.320891708463282
Department of Science and Technology (Philippines)

The Philippines' Department of Science and Technology (abbreviated as DOST; ), is the executive department of the Philippine Government responsible for the coordination of science and technology-related projects in the Philippines and to formulate 
--------------------------------------------------------------------------------
Documento: doc_1317
Score: 7.306869367836512
Ministry of Science and Technology (Taiwan)

The Ministry of Science and Technology (MOST; ) is the go

## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [26]:
queries = [
    "science and technology",
    "love and friendship",
    "war and history",
    "religion and faith",
    "nature and environment",
    "education and learning",
    "economy and money",
    "health and medicine",
    "art and culture",
    "computer and internet"
]

In [27]:
for query in queries:

    print("=" * 80)
    print("QUERY:", query)

    idx_tfidf, _ = buscar_tfidf(query)

    resultados_bm25 = buscar_bm25_manual(query)

    print("\nTF-IDF:")
    for i in idx_tfidf:
        print("-", nombres[i])

    print("\nBM25:")

    for idx, score in resultados_bm25:
        print("-", nombres[idx], "| Score:", round(score, 4))

QUERY: science and technology

TF-IDF:
- doc_3842
- doc_1317
- doc_6377
- doc_5501
- doc_828

BM25:
- doc_5501 | Score: 7.4208
- doc_6377 | Score: 7.3209
- doc_1317 | Score: 7.3069
- doc_5493 | Score: 7.2677
- doc_9026 | Score: 7.2381
QUERY: love and friendship

TF-IDF:
- doc_4261
- doc_8008
- doc_4752
- doc_3610
- doc_3438

BM25:
- doc_3438 | Score: 13.6086
- doc_4752 | Score: 10.6812
- doc_4261 | Score: 9.9774
- doc_8942 | Score: 9.0261
- doc_8008 | Score: 8.7547
QUERY: war and history

TF-IDF:
- doc_9265
- doc_6079
- doc_1600
- doc_9024
- doc_3981

BM25:
- doc_9265 | Score: 10.7198
- doc_3164 | Score: 9.4614
- doc_1061 | Score: 9.2142
- doc_9864 | Score: 9.0729
- doc_5161 | Score: 8.7527
QUERY: religion and faith

TF-IDF:
- doc_8098
- doc_3334
- doc_9392
- doc_1954
- doc_1951

BM25:
- doc_9588 | Score: 14.4896
- doc_8098 | Score: 11.2484
- doc_9392 | Score: 10.385
- doc_7269 | Score: 10.303
- doc_1951 | Score: 9.9754
QUERY: nature and environment

TF-IDF:
- doc_5440
- doc_1809
- doc